# Data Quality Tests for Wheelie Data Warehouse

This notebook contains data quality tests for all dimension tables and bridge tables.
Each test validates uniqueness constraints and referential integrity.

In [ ]:
!pip install pytest

In [ ]:
# reload imports
dbutils.library.restartPython()

In [ ]:
# ==============================================================================
# TEST CONFIGURATION
# ==============================================================================
import pytest
import logging
from pyspark.sql.functions import col, count, countDistinct

logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(message)s')
logger = logging.getLogger("data_quality_tests")

def get_table(table_name: str):
    """Helper to load table from data warehouse."""
    return spark.table(f"wheelie.data_warehouse.{table_name}")

def assert_uniqueness(table_name: str, column_name: str):
    """Assert that a column has all unique values."""
    df = get_table(table_name)
    total_count = df.count()
    distinct_count = df.select(column_name).distinct().count()
    duplicates_count = total_count - distinct_count

    if duplicates_count > 0:
        duplicates = df.groupBy(column_name).count().filter(col("count") > 1)
        logger.error(f"Duplicate values found in {table_name}.{column_name}:")
        duplicates.show(10)

    assert total_count == distinct_count, \
        f"{table_name}.{column_name}: Found {duplicates_count} duplicate(s). Expected all {total_count} values to be unique."

def assert_referential_integrity(source_table: str, source_col: str, target_table: str, target_col: str):
    """Assert that all values in source column exist in target column."""
    source_df = get_table(source_table)
    target_df = get_table(target_table)

    missing = source_df.select(col(source_col).alias("key")) \
        .distinct() \
        .join(
            target_df.select(col(target_col).alias("key")),
            "key",
            "left_anti"
        )

    missing_count = missing.count()
    total_count = source_df.select(source_col).distinct().count()

    if missing_count > 0:
        logger.error(f"Missing references from {source_table}.{source_col} to {target_table}.{target_col}:")
        missing.show(10)

    assert missing_count == 0, \
        f"Referential integrity violation: {missing_count} out of {total_count} keys in {source_table}.{source_col} do not exist in {target_table}.{target_col}"

logger.info("=" * 70)
logger.info("DATA QUALITY TESTS STARTED")
logger.info("=" * 70)


In [ ]:
# ==============================================================================
# TEST: BRIDGE_CAR_EQUIPMENT
# ==============================================================================

def test_bridge_car_equipment_car_key_unique():
    """Test that car_key is unique in bridge_car_equipment."""
    logger.info("Testing: bridge_car_equipment.car_key uniqueness")
    assert_uniqueness("bridge_car_equipment", "car_key")
    logger.info("✅ PASS: bridge_car_equipment.car_key is unique")


In [ ]:
# ==============================================================================
# TEST: BRIDGE_EQUIPMENT_GROUP_EQUIPMENT
# ==============================================================================

def test_bridge_equipment_group_equipment_group_key_exists():
    """Test that all equipment_group_key values exist in bridge_car_equipment."""
    logger.info("Testing: bridge_equipment_group_equipment.equipment_group_key referential integrity")
    assert_referential_integrity(
        source_table="bridge_equipment_group_equipment",
        source_col="equipment_group_key",
        target_table="bridge_car_equipment",
        target_col="equipment_group_key"
    )
    logger.info("✅ PASS: All equipment_group_key exist in bridge_car_equipment")

def test_bridge_equipment_group_equipment_key_exists():
    """Test that all equipment_key values exist in dim_equipment."""
    logger.info("Testing: bridge_equipment_group_equipment.equipment_key referential integrity")
    assert_referential_integrity(
        source_table="bridge_equipment_group_equipment",
        source_col="equipment_key",
        target_table="dim_equipment",
        target_col="equipment_key"
    )
    logger.info("✅ PASS: All equipment_key exist in dim_equipment")



In [ ]:
# ==============================================================================
# TEST: DIM_CAR
# ==============================================================================

def test_dim_car_car_key_unique():
    """Test that car_key is unique in dim_car."""
    logger.info("Testing: dim_car.car_key uniqueness")
    assert_uniqueness("dim_car", "car_key")
    logger.info("✅ PASS: dim_car.car_key is unique")

test_dim_car_car_key_unique()


In [ ]:
# ==============================================================================
# TEST: DIM_CUSTOMER
# ==============================================================================

def test_dim_customer_customer_key_unique():
    """Test that customer_key is unique in dim_customer."""
    logger.info("Testing: dim_customer.customer_key uniqueness")
    assert_uniqueness("dim_customer", "customer_key")
    logger.info("✅ PASS: dim_customer.customer_key is unique")

def test_dim_customer_customer_id_unique():
    """Test that customer_id is unique in dim_customer."""
    logger.info("Testing: dim_customer.customer_id uniqueness")
    assert_uniqueness("dim_customer", "customer_id")
    logger.info("✅ PASS: dim_customer.customer_id is unique")



In [ ]:
# ==============================================================================
# TEST: DIM_STAFF
# ==============================================================================

def test_dim_staff_staff_key_unique():
    """Test that staff_key is unique in dim_staff."""
    logger.info("Testing: dim_staff.staff_key uniqueness")
    assert_uniqueness("dim_staff", "staff_key")
    logger.info("✅ PASS: dim_staff.staff_key is unique")

test_dim_staff_staff_key_unique()


In [ ]:
# ==============================================================================
# TEST: DIM_STORE
# ==============================================================================

def test_dim_store_store_key_unique():
    """Test that store_key is unique in dim_store."""
    logger.info("Testing: dim_store.store_key uniqueness")
    assert_uniqueness("dim_store", "store_key")
    logger.info("✅ PASS: dim_store.store_key is unique")

def test_dim_store_store_id_unique():
    """Test that store_id is unique in dim_store."""
    logger.info("Testing: dim_store.store_id uniqueness")
    assert_uniqueness("dim_store", "store_id")
    logger.info("✅ PASS: dim_store.store_id is unique")


In [ ]:
# ==============================================================================
# TEST SUMMARY
# ==============================================================================

logger.info("\n" + "=" * 70)
logger.info("TEST SUMMARY")
logger.info("=" * 70)

# Collect all test functions
test_functions = [
    test_bridge_car_equipment_car_key_unique,
    test_bridge_equipment_group_equipment_group_key_exists,
    test_bridge_equipment_group_equipment_key_exists,
    test_dim_car_car_key_unique,
    test_dim_customer_customer_key_unique,
    test_dim_customer_customer_id_unique,
    test_dim_staff_staff_key_unique,
    test_dim_store_store_key_unique,
    test_dim_store_store_id_unique,
]

passed = 0
failed = 0
failed_tests = []

for test_func in test_functions:
    try:
        test_func()
        passed += 1
    except AssertionError as e:
        failed += 1
        failed_tests.append({
            "test": test_func.__name__,
            "description": test_func.__doc__,
            "error": str(e)
        })
        logger.error(f"❌ FAILED: {test_func.__name__}")
        logger.error(f"   {str(e)}")

total = len(test_functions)

logger.info(f"\nTotal Tests: {total}")
logger.info(f"Passed: {passed} ✅")
logger.info(f"Failed: {failed} ❌")

if failed > 0:
    logger.error("\n⚠️  Failed Tests:")
    for test in failed_tests:
        logger.error(f"  - {test['test']}: {test['description']}")
        logger.error(f"    Error: {test['error']}")
    logger.warning(f"\n⚠️  {failed} test(s) FAILED - review the results above")
else:
    logger.info("\n🎉 All data quality tests PASSED!")
